# Post 1: the heavier ResNets, on a Colab TPU

The CIFAR-10 runs that are too slow for a laptop CPU. Everything else about them
matches `01_data_and_pipeline_cifar10.ipynb`: same augmentation, AdamW with weight
decay 0.1, cosine learning-rate decay, and a batch of 128 however many devices share
it, so the results compare directly with the earlier runs.

Metrics and checkpoints go to Weights & Biases as well as the runtime's disk. A Colab
runtime can vanish at any time, and its disk goes with it. **If that happens, connect
to a new runtime and run every cell again: each run carries on from its last
uploaded checkpoint, and finished runs are skipped.**

See `COLAB.md` for how to connect this notebook to a Colab runtime from your editor.

## Setup

On Colab this clones the branch below and installs it. `BRANCH` must already be pushed,
because the runtime sees GitHub, not your working copy. Locally it does nothing.

The repo needs a JAX recent enough to have `jax.set_mesh`. If the runtime's is older,
this upgrades it before anything imports JAX.

In [ ]:
import os
import pathlib
import sys

BRANCH = "p01-data-pipeline"

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("/content/toylib").exists():
        !git clone --depth 1 --branch {BRANCH} https://github.com/anujkhare/toylib.git /content/toylib
    %cd /content/toylib
    !git pull --ff-only origin {BRANCH}
    !pip install -q -e ".[harness,viz]"
    has_set_mesh = !python -c "import jax; print(hasattr(jax, 'set_mesh'))"
    if has_set_mesh[-1] != "True":
        !pip install -q -U "jax[tpu]"

## Sign in to Weights & Biases

Paste your key from <https://wandb.ai/authorize> when asked. The prompt keeps it out of
the notebook. Colab's `userdata` secrets don't work from the VS Code extension, so they
aren't an option here. Locally, `wandb login` has already stored it.

In [ ]:
import getpass

WANDB_PROJECT = "toylib-p01-cifar10"
WANDB_ENTITY = ""  # Blank uses your account's default.

if IN_COLAB and "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_API_KEY"] = getpass.getpass("W&B API key: ")

## The runs

`blocks_per_stage=n` gives `6n + 2` layers. All three run in this order, one at a time,
since they share one accelerator:

| Run | Layers | Width | Why |
| --- | --- | --- | --- |
| `resnet56` | 56 | 16 | Does depth keep helping? It reached 78.2% in the 15-epoch CPU run. |
| `plain56` | 56 | 16 | The same network with no skip connections. It reached 34.3% in 15 epochs, and this shows whether it was only slow or truly stuck. |
| `resnet20w64` | 20 | 64 | Does width help more than depth? Four times the width of ResNet-20, about 16 times the parameters. |

Evaluation batches are 400, not the 500 used elsewhere: the harness needs a batch to
divide evenly across devices, and an 8-core TPU can't split 500.

In [ ]:
import jax
import numpy as np
import optax
from absl import logging as absl_logging

from toylib.harness import analyze
from toylib.harness.experiment import (
    CheckpointConfig,
    EvalConfig,
    Experiment,
    LoggerConfig,
    MultiOptimizerConfig,
    OptimizerConfig,
    Task,
    TrainingConfig,
    WandBLoggerConfig,
)
from toylib.harness.metrics import Loss
from toylib_projects.p01_cnn_mnist import data, metrics, model, resnet

RUNS = {
    "resnet56": resnet.ResNetConfig(blocks_per_stage=9),
    "plain56": resnet.ResNetConfig(blocks_per_stage=9, residual=False),
    "resnet20w64": resnet.ResNetConfig(width=64),
}
EPOCHS = 40
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 400
WEIGHT_DECAY = 0.1
CHECKPOINT_EVERY_EPOCHS = 5  # The most work a lost runtime can cost.

absl_logging.set_verbosity(absl_logging.ERROR)  # quiet orbax's read-only notices
print("JAX", jax.__version__, "| devices:", jax.devices())
if jax.default_backend() == "cpu":
    print("WARNING: no accelerator. Check the runtime type before starting.")

In [ ]:
def make_experiment(name, max_steps=None, logger_config=None, out_dir=None):
    """One run through the harness. Defaults to the full run, logged to W&B."""
    train_ds, test_ds = data.make_datasets(
        "cifar10", batch_size=BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE, augment=True
    )
    steps_per_epoch = train_ds.batches_per_epoch
    total_steps = EPOCHS * steps_per_epoch
    run_id = f"{name}_e{EPOCHS}"
    out_dir = out_dir or pathlib.Path(model.__file__).parent / "outputs" / f"resnet_{run_id}"

    learning_rate = optax.cosine_decay_schedule(1e-3, total_steps)
    optimizer_config = MultiOptimizerConfig(
        [OptimizerConfig("adamw", optax.adamw(learning_rate, weight_decay=WEIGHT_DECAY))],
        optimizer_for_param=lambda path: "adamw",
    )
    task_metrics = [Loss(), metrics.Accuracy()]
    return Experiment(
        train_task=Task("train", train_ds, metrics=task_metrics),
        eval_task=Task("test", test_ds, metrics=task_metrics),
        model_factory=resnet.build_resnet,
        forward_fn=model.forward,
        model_config=RUNS[name],
        # +1: the harness evaluates and saves a step before counting it, so this
        # makes the last epoch's evaluation and checkpoint happen.
        training_config=TrainingConfig(
            optimizer_config=optimizer_config, max_steps=max_steps or total_steps + 1
        ),
        eval_config=EvalConfig(
            eval_interval_steps=steps_per_epoch, num_eval_steps=test_ds.batches_per_epoch
        ),
        checkpoint_config=CheckpointConfig(
            save_interval_steps=CHECKPOINT_EVERY_EPOCHS * steps_per_epoch,
            max_to_keep=2,
            checkpoint_dir=str(out_dir / "checkpoints"),
            checkpoint_dataset_iterator=True,  # Resume mid-epoch with the same order.
        ),
        logger_config=logger_config
        or WandBLoggerConfig(
            project_name=WANDB_PROJECT,
            user_name=WANDB_ENTITY,
            log_dir=str(out_dir),
            run_id=run_id,  # A repeat of this run resumes the same W&B run.
            train_log_interval_steps=50,
            upload_checkpoints=True,
            also_log_to_file=True,
        ),
    )

## How long will this take?

A short timing run of each model, logged nowhere: 100 training steps and one evaluation
after compiling. It also proves that the model, sharding and evaluation work on this
accelerator before hours go into them. A step here is the same size as a real step, so
the estimate covers the whole run, including one evaluation per epoch.

In [ ]:
import tempfile
import time

estimates = {}
for name, config in RUNS.items():
    with tempfile.TemporaryDirectory() as tmp:
        tmp = pathlib.Path(tmp)
        exp = make_experiment(
            name, logger_config=LoggerConfig(log_dir=str(tmp), run_id="timing"), out_dir=tmp
        )
        exp.init_state()
        batches = iter(exp.train_task.dataset)
        exp.inner_loop(next(batches))  # Compiles the training step.
        jax.block_until_ready(exp.model)

        start = time.monotonic()
        for _ in range(100):
            exp.inner_loop(next(batches))
        jax.block_until_ready(exp.model)
        seconds_per_step = (time.monotonic() - start) / 100

        exp.run_validation()  # Compiles the evaluation step.
        start = time.monotonic()
        exp.run_validation()
        seconds_per_eval = time.monotonic() - start

        n_params = analyze.print_param_sizes(exp.model, depth=1)[1]
        steps = exp.training_config.max_steps
        hours = (steps * seconds_per_step + EPOCHS * seconds_per_eval) / 3600
        exp.cleanup()
    estimates[name] = hours
    print(f"{name:12s} {n_params:>10,} params  {1 / seconds_per_step:6.1f} steps/s  "
          f"{seconds_per_eval:5.1f} s/eval  about {hours:.2f} h for {EPOCHS} epochs")
print(f"total: about {sum(estimates.values()):.2f} h")

## Train

Prints a W&B link for each run, where the curves appear as it trains. Each run starts
from its newest checkpoint if it has one, on this runtime or uploaded to W&B. Keep the
editor open and the laptop awake while this runs.

In [ ]:
results = {}
for name in RUNS:
    exp = make_experiment(name)
    try:
        exp.init_state()
        resumed = exp.resume()
        print(f"\n{name}: " + (f"resuming at step {exp.step}" if resumed else "starting fresh")
              + f" of {exp.training_config.max_steps}")
        exp.outer_loop()
        results[name] = exp.run_validation()
    finally:
        exp.cleanup()  # Waits for checkpoint uploads to finish.
    print(f"{name}: test accuracy {results[name]['val/accuracy']:.4f}, "
          f"test loss {results[name]['val/loss']:.4f}")

## Results

Paste this table back into the session, or compare the runs on W&B. For reference, the
40-epoch CPU runs reached 85.0% with ResNet-20, 76.4% with ResNet-8 and 73.3% with Plain-20.
Checkpoints stay on W&B as the artifacts `<run>_e40-checkpoints`.

In [ ]:
for name, result in results.items():
    print(f"{name:12s} test accuracy {result['val/accuracy']:.4f}  test loss {result['val/loss']:.4f}")